In [ ]:
import re, json, random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, f1_score, confusion_matrix, precision_recall_curve

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
LABELS = ["toxicity", "hate", "harassment", "abuse"]

JIGSAW_PATH = "train.csv"

MAX_LEN = 100
VOCAB_SIZE = 20000
EMBED_DIM = 128
HIDDEN_DIM = 128
DROPOUT = 0.3
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
MAX_EPOCHS = 15
PATIENCE = 3
FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0

N_SYNTHETIC_HARD_NEGATIVES = 4000

MODEL_PATH = "toxicity_model_v3.pt"
TOKENIZER_PATH = "bpe_tokenizer.json"
THRESHOLDS_PATH = "thresholds.json"
CONFIG_PATH = "model_config_v3.json"


In [ ]:
jigsaw_df = pd.read_csv(JIGSAW_PATH)

jigsaw_df["toxicity"]   = ((jigsaw_df["toxic"] == 1) | (jigsaw_df["severe_toxic"] == 1)).astype(int)
jigsaw_df["hate"]       = jigsaw_df["identity_hate"].astype(int)
jigsaw_df["harassment"] = ((jigsaw_df["insult"] == 1) | (jigsaw_df["threat"] == 1)).astype(int)
jigsaw_df["abuse"]      = ((jigsaw_df["obscene"] == 1) | (jigsaw_df["severe_toxic"] == 1)).astype(int)

jigsaw_df = jigsaw_df.rename(columns={"comment_text": "text"})
jigsaw_df = jigsaw_df[["text"] + LABELS].dropna().drop_duplicates(subset="text")
jigsaw_df["label_mask"] = [[1, 1, 1, 1]] * len(jigsaw_df)
jigsaw_df["source"] = "jigsaw_en"

print("jigsaw rows:", len(jigsaw_df))
print(jigsaw_df[LABELS].mean())

In [ ]:
def mine_toxic_words(df, label="toxicity", top_k=150, min_count=20):
    pos_texts = df.loc[df[label] == 1, "text"].str.lower().str.split()
    neg_texts = df.loc[df[label] == 0, "text"].str.lower().str.split()
    pos_counts = Counter(w for toks in pos_texts for w in toks)
    neg_counts = Counter(w for toks in neg_texts for w in toks)
    scored = []
    for w, c in pos_counts.items():
        if c < min_count or not w.isalpha() or len(w) < 3:
            continue
        ratio = c / (neg_counts.get(w, 0) + 1)
        scored.append((ratio, w))
    scored.sort(reverse=True)
    return [w for _, w in scored[:top_k]]


NEGATION_TEMPLATES = [
    "I don't think you're {w} at all.",
    "That's not {w}, honestly.",
    "You're absolutely not {w}.",
    "No one would call that {w}.",
    "I wouldn't say this is {w}.",
    "Some people think this is {w}, but I disagree.",
    "This used to be considered {w}, but not anymore.",
    "Not {w}, just direct.",
]


def generate_hard_negatives(df, n=N_SYNTHETIC_HARD_NEGATIVES, seed=SEED):
    rng = random.Random(seed)
    words = mine_toxic_words(df)
    rows = [rng.choice(NEGATION_TEMPLATES).format(w=rng.choice(words)) for _ in range(n)]
    out = pd.DataFrame({"text": rows}).drop_duplicates(subset="text")
    for lbl in LABELS:
        out[lbl] = 0
    out["label_mask"] = [[1, 1, 1, 1]] * len(out)
    out["source"] = "synthetic_hard_negative"
    return out


synthetic_df = generate_hard_negatives(jigsaw_df)
print("synthetic hard negatives:", len(synthetic_df))
synthetic_df.head()

In [ ]:
combined_df = pd.concat([jigsaw_df, synthetic_df], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("combined rows:", len(combined_df))
print(combined_df["source"].value_counts())
print(combined_df[LABELS].mean())

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
WS_RE = re.compile(r"\s+")

def clean_text(text):
    text = URL_RE.sub(" <URL> ", text)
    text = WS_RE.sub(" ", text).strip()
    return text

combined_df["clean_text"] = combined_df["text"].astype(str).apply(clean_text)
combined_df[["clean_text", "source"]].head()

In [ ]:
combined_df["char_len"] = combined_df["clean_text"].str.len()
print(combined_df["char_len"].describe())
print("share over 500 chars:", (combined_df["char_len"] > 500).mean())

In [ ]:
train_df, temp_df = train_test_split(combined_df, train_size=0.70, random_state=SEED)
val_df, test_df = train_test_split(temp_df, train_size=0.5, random_state=SEED)
len(train_df), len(val_df), len(test_df)

In [ ]:
from tokenizers import ByteLevelBPETokenizer

with open("bpe_train_corpus.txt", "w", encoding="utf-8") as f:
    for t in train_df["clean_text"]:
        f.write(t.replace("\n", " ") + "\n")

bpe = ByteLevelBPETokenizer()
bpe.train(
    files=["bpe_train_corpus.txt"],
    vocab_size=VOCAB_SIZE,
    min_frequency=2,
    special_tokens=["<PAD>", "<UNK>", "<URL>"],
)
bpe.save(TOKENIZER_PATH)

PAD_ID = bpe.token_to_id("<PAD>")
UNK_ID = bpe.token_to_id("<UNK>")

def encode(text, max_len=MAX_LEN):
    ids = bpe.encode(text).ids[:max_len]
    ids += [PAD_ID] * (max_len - len(ids))
    return ids

print("BPE vocab size:", bpe.get_vocab_size())
print("sample encode:", bpe.encode("you're not an idiot at all").tokens)

In [ ]:
class ToxicityDataset(Dataset):
    def __init__(self, dframe):
        self.texts = dframe["clean_text"].tolist()
        self.labels = dframe[LABELS].values.astype("float32")
        self.masks = np.stack(dframe["label_mask"].values).astype("float32")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        ids = encode(self.texts[idx])
        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(self.labels[idx]),
            torch.tensor(self.masks[idx]),
        )

train_ds = ToxicityDataset(train_df)
val_ds   = ToxicityDataset(val_df)
test_ds  = ToxicityDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE)

# sanity check — confirm 3 values per batch, not 4
xb, yb, mb = next(iter(train_loader))
xb.shape, yb.shape, mb.shape

In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_out, pad_mask):
        scores = self.attn(lstm_out).squeeze(-1)
        scores = scores.masked_fill(pad_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_out).squeeze(1)
        return context, weights


class BiLSTMAttnToxicityClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels, dropout=0.3, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attention = Attention(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, x, return_attention=False):
        pad_mask = (x != self.pad_idx).float()
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        context, attn_weights = self.attention(lstm_out, pad_mask)
        context = self.dropout(context)
        logits = self.fc(context)
        if return_attention:
            return logits, attn_weights
        return logits

    def encode(self, x):
        """Attention context vector, exposed for reuse (e.g. the /suggest
        retrieval feature) without going through the classification head."""
        pad_mask = (x != self.pad_idx).float()
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        context, _ = self.attention(lstm_out, pad_mask)
        return context


model = BiLSTMAttnToxicityClassifier(bpe.get_vocab_size(), EMBED_DIM, HIDDEN_DIM, len(LABELS), pad_idx=PAD_ID).to(device)
model

In [ ]:
def masked_focal_loss(logits, targets, mask, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
    probs = torch.sigmoid(logits)
    ce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = probs * targets + (1 - probs) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    loss = alpha_t * (1 - p_t).pow(gamma) * ce
    loss = loss * mask
    return loss.sum() / mask.sum().clamp(min=1.0)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
best_val_macro_f1 = -1.0
patience_counter = 0

for epoch in range(MAX_EPOCHS):
    model.train()
    train_loss = 0.0
    for x, y, m in train_loader:
        x, y, m = x.to(device), y.to(device), m.to(device)
        optimizer.zero_grad()
        loss = masked_focal_loss(model(x), y, m)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= len(train_ds)

    model.eval()
    val_probs, val_true = [], []
    with torch.no_grad():
        for x, y, m in val_loader:
            x = x.to(device)
            val_probs.append(torch.sigmoid(model(x)).cpu())
            val_true.append(y)
    val_probs = torch.cat(val_probs).numpy()
    val_true = torch.cat(val_true).numpy()
    val_preds = (val_probs >= 0.5).astype(int)
    val_macro_f1 = f1_score(val_true, val_preds, average="macro", zero_division=0)

    print(f"epoch {epoch+1}: train_loss={train_loss:.4f} val_macro_f1={val_macro_f1:.4f}")

    if val_macro_f1 > best_val_macro_f1:
        best_val_macro_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_PATH)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

model.load_state_dict(torch.load(MODEL_PATH))
print("Best val macro F1:", best_val_macro_f1)

In [ ]:
def get_probs(loader):
    model.eval()
    all_probs, all_true, all_masks = [], [], []
    with torch.no_grad():
        for x, y, m in loader:
            x = x.to(device)
            probs = torch.sigmoid(model(x)).cpu()
            all_probs.append(probs)
            all_true.append(y)
            all_masks.append(m)
    return torch.cat(all_probs).numpy(), torch.cat(all_true).numpy(), torch.cat(all_masks).numpy()

val_probs, val_true, val_mask = get_probs(val_loader)

thresholds = {}
for i, label in enumerate(LABELS):
    valid = val_mask[:, i] == 1
    p, r, t = precision_recall_curve(val_true[valid, i], val_probs[valid, i])
    f1 = 2 * p * r / (p + r + 1e-9)
    best_idx = f1[:-1].argmax() if len(t) > 0 else 0
    thresholds[label] = float(t[best_idx]) if len(t) > 0 else 0.5

with open(THRESHOLDS_PATH, "w") as f:
    json.dump(thresholds, f)

thresholds

In [ ]:
test_probs, test_true, test_mask = get_probs(test_loader)

for i, label in enumerate(LABELS):
    valid = test_mask[:, i] == 1
    preds = (test_probs[valid, i] >= thresholds[label]).astype(int)
    p, r, f, _ = precision_recall_fscore_support(test_true[valid, i], preds, average="binary", zero_division=0)
    cm = confusion_matrix(test_true[valid, i], preds)
    fn = int(cm[1][0]) if cm.shape == (2, 2) else None
    print(f"{label:12s} n={valid.sum():>7d} thr={thresholds[label]:.2f} precision={p:.3f} recall={r:.3f} f1={f:.3f} false_negatives={fn}")

all_preds = np.stack([(test_probs[:, i] >= thresholds[label]).astype(int) for i, label in enumerate(LABELS)], axis=1)
print("\ntest macro F1:", f1_score(test_true, all_preds, average="macro", zero_division=0))

In [ ]:
def predict(text, top_n=5):
    cleaned = clean_text(text)
    ids = encode(cleaned)
    x = torch.tensor([ids], dtype=torch.long).to(device)

    model.eval()
    with torch.no_grad():
        logits, attn = model(x, return_attention=True)
        probs = torch.sigmoid(logits)[0].cpu().tolist()

    per_label = {}
    for label, p in zip(LABELS, probs):
        per_label[label] = {"prob": round(p, 4), "flagged": bool(p >= thresholds[label])}

    overall = 1 - float(np.prod([1 - p for p in probs]))

    tokens = bpe.encode(cleaned).tokens[:MAX_LEN]
    weights = attn[0][: len(tokens)].cpu().tolist()
    top_tokens = sorted(zip(tokens, weights), key=lambda t: -t[1])[:top_n]

    return {
        "per_label": per_label,
        "overall_toxicity": round(overall, 4),
        "any_flagged": any(v["flagged"] for v in per_label.values()),
        "top_attended_tokens": [{"token": t, "weight": round(w, 4)} for t, w in top_tokens],
    }

for example in [
    "I really enjoyed this movie.",
    "You are a disgusting idiot.",
    "I don't think you're an idiot.",
    "xoxo ur so dum lol",
]:
    print(example, "->", predict(example))

In [ ]:
with open(CONFIG_PATH, "w") as f:
    json.dump({
        "vocab_size": bpe.get_vocab_size(), "embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM,
        "num_labels": len(LABELS), "dropout": DROPOUT, "max_len": MAX_LEN, "labels": LABELS,
        "focal_alpha": FOCAL_ALPHA, "focal_gamma": FOCAL_GAMMA,
    }, f)

print("Saved:", MODEL_PATH, TOKENIZER_PATH, THRESHOLDS_PATH, CONFIG_PATH)

In [ ]:
def get_toxicity_level(toxicity_score):
    if toxicity_score < 0.60:
        return {"level": "safe", "blur": 0, "message": "No blur required."}
    elif toxicity_score < 0.85:
        return {"level": "moderate", "blur": 1, "message": "Content temporarily blurred due to potentially harmful language."}
    else:
        return {"level": "high", "blur": 2, "message": "Content heavily blurred due to highly toxic or harmful language."}


def get_toxicity_warning(toxicity_score):
    if toxicity_score < 0.60:
        return {"show_warning": False, "title": "Content appears safe",
                "reason": "The detected toxicity level is below the warning threshold."}
    elif toxicity_score < 0.85:
        return {"show_warning": True, "title": "Potentially harmful content",
                "reason": "This content may contain offensive, abusive, or inappropriate language. It has been temporarily blurred."}
    else:
        return {"show_warning": True, "title": "Highly toxic content",
                "reason": "This content has a high probability of containing severely offensive, abusive, threatening, or harmful language. It has been heavily blurred for safety."}


def get_age_rating(toxicity_score):
    if toxicity_score < 0.60:
        return {"age_rating": "13-15", "level": "Low"}
    elif toxicity_score < 0.70:
        return {"age_rating": "15-18", "level": "Moderate"}
    elif toxicity_score < 0.85:
        return {"age_rating": "18-21", "level": "High"}
    else:
        return {"age_rating": "21+", "level": "Very High"}


def build_rating_payload(toxicity_score):
    tox_level = get_toxicity_level(toxicity_score)
    warning = get_toxicity_warning(toxicity_score)
    age = get_age_rating(toxicity_score)
    return {
        "ageRating": age["age_rating"],
        "is_sensitive": warning["show_warning"],
        "toxicity_rating": tox_level["level"],
        "message": tox_level["message"],
        "blur_level": tox_level["blur"],
        "warning_title": warning["title"],
        "warning_reason": warning["reason"],
        "age_rating_level": age["level"],
    }

build_rating_payload(0.78)